# Tutorial 5: Inference Methods — MAP, NUTS, and geoVI

diffsed supports three gradient-based inference methods. All exploit end-to-end differentiability through the forward model.

| Method | Speed | Uncertainties | Best for |
|--------|-------|---------------|----------|
| **MAP** (Adam) | Seconds | No | Large surveys, initial guesses |
| **NUTS** (BlackJAX) | Minutes | Yes (exact) | Gold-standard posteriors |
| **geoVI** (NIFTy.re) | Seconds | Yes (approx) | Fast posteriors, hierarchical models |

In [ ]:
%matplotlib inline
import sys; sys.path.insert(0, "../src")
import warnings; warnings.filterwarnings('ignore')
import logging; logging.disable(logging.WARNING)

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import time, io

from diffsed import (
    Model, ParamSpec, Fitter, Uniform, Gaussian, Fixed,
    load_ssp_data, load_filter_set,
)

plt.rcParams.update({
    "figure.dpi": 130, "font.size": 11,
    "axes.linewidth": 1.0,
    "xtick.direction": "in", "ytick.direction": "in",
    "xtick.top": True, "ytick.right": True,
})

ssp = load_ssp_data("../data/ssp_prsc_miles_chabrier_wNE_logGasU-3.0_logGasZ0.0.h5")
filters = load_filter_set(["sdss_u", "sdss_g", "sdss_r", "sdss_i", "sdss_z"])
filter_names = ["u", "g", "r", "i", "z"]

## Setup: Mock Galaxy at z = 0.1

We fit a single parametric mock galaxy (no stochastic GP) with all three methods and compare.

In [ ]:
spec = ParamSpec(
    sfh_alpha        = Uniform(0.5, 3.0),
    sfh_beta         = Uniform(0.3, 2.0),
    sfh_tau_peak_gyr = Uniform(0.5, 10.0),
    sfh_peak_sfr     = Uniform(0.1, 50.0),
    met_logzsol      = Uniform(-1.5, 0.2),
    dust_tau_bc      = Uniform(0.0, 3.0),
    dust_tau_diff    = Uniform(0.0, 2.0),
    dust_slope       = -0.7,
    redshift         = 0.1,
    stochastic       = False,
)
model = Model(spec, ssp, filters=filters)

true_params = {
    "sfh_alpha": 1.5, "sfh_beta": 1.0,
    "sfh_tau_peak_gyr": 4.0, "sfh_peak_sfr": 8.0,
    "met_logzsol": -0.3,
    "dust_tau_bc": 1.0, "dust_tau_diff": 0.3,
    "dust_slope": -0.7, "redshift": 0.1,
}

mock = model.mock(true_params, snr=20.0, key=jax.random.PRNGKey(0))
sfh_true = model.predict_sfh(true_params)

print(f"Mock: {len(filter_names)} bands, SNR=20, z={true_params['redshift']}")
print(f"Free parameters: {spec.n_free}")

---
## 1. MAP (Adam)

Fast point estimate — seconds per galaxy. No error bars.

In [ ]:
fitter = Fitter(model, mock.flux_obs, mock.noise)

result_map = fitter.run("map", n_steps=2000, learning_rate=0.03, verbose=True)
print(f"\nWall time: {result_map.wall_time_s:.1f}s")

---
## 2. NUTS (BlackJAX)

Exact posterior via Hamiltonian Monte Carlo. Uses the gradient to efficiently explore the posterior surface. Initialized from MAP for faster convergence.

In [ ]:
try:
    result_nuts = fitter.run(
        "nuts", init_from=result_map,
        n_warmup=300, n_samples=300,
        target_accept_rate=0.85, verbose=True,
    )
    nuts_ok = True
    print(f"\nDivergences: {result_nuts.diagnostics['n_divergent']}/{result_nuts.diagnostics['n_samples']}")
except Exception as e:
    nuts_ok = False
    print(f"NUTS failed: {type(e).__name__}: {str(e)[:200]}")

---
## 3. geoVI (NIFTy.re)

Approximate posterior via geometric variational inference. Finds a coordinate transform where the posterior is approximately Gaussian. Much faster than NUTS with comparable quality for most SED fits.

In [ ]:
try:
    # Suppress NIFTy verbose output
    old_stdout = sys.stdout
    sys.stdout = io.StringIO()
    
    result_geovi = fitter.run(
        "geovi", init_from=result_map,
        n_iterations=15, n_samples=6, n_posterior_samples=40, verbose=False,
    )
    sys.stdout = old_stdout
    geovi_ok = True
    print(f"geoVI: {result_geovi.diagnostics['n_samples']} samples in {result_geovi.wall_time_s:.1f}s")
except Exception as e:
    sys.stdout = old_stdout
    geovi_ok = False
    print(f"geoVI failed: {type(e).__name__}: {str(e)[:200]}")

---
## 4. Comparison: SFH Recovery (Linear Scale)

The key comparison: how well does each method recover the true star formation history? Plotted in **linear** SFR for intuitive interpretation.

In [ ]:
methods = {"MAP": (result_map, "C0")}
if nuts_ok:
    methods["NUTS"] = (result_nuts, "C1")
if geovi_ok:
    methods["geoVI"] = (result_geovi, "C2")

n_methods = len(methods)
fig, axes = plt.subplots(1, n_methods, figsize=(5 * n_methods, 5), sharey=True)
if n_methods == 1: axes = [axes]

for ax, (name, (res, color)) in zip(axes, methods.items()):
    model.plot_sfh_posterior(res, true_params=true_params, ax=ax,
                            color=color, label=name)
    ax.set_title(f"{name} ({res.wall_time_s:.1f}s)")
    ax.text(0.98, 0.98, f"z = {true_params['redshift']}", transform=ax.transAxes,
            ha="right", va="top", fontsize=9, style="italic")

plt.suptitle("SFH Recovery — Linear Scale (16-84% fill)", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

---
## 5. Parameter Posteriors (Physical Units)

In [ ]:
# Show posteriors for methods that have samples
sampling_results = {}
if nuts_ok:
    sampling_results["NUTS"] = (result_nuts, "C1")
if geovi_ok:
    sampling_results["geoVI"] = (result_geovi, "C2")

if sampling_results:
    phys_params = [
        ("sfh_alpha",         r"$\alpha$",                 1.5),
        ("sfh_beta",          r"$\beta$",                  1.0),
        ("sfh_tau_peak_gyr",  r"$\tau_{\rm peak}$ (Gyr)", 4.0),
        ("sfh_peak_sfr",      r"SFR$_{\rm peak}$",        8.0),
        ("met_logzsol",       r"log Z",                    -0.3),
        ("dust_tau_bc",       r"$\tau_{\rm bc}$",         1.0),
        ("dust_tau_diff",     r"$\tau_{\rm diff}$",       0.3),
    ]

    n_params = len(phys_params)
    fig, axes_p = plt.subplots(2, 4, figsize=(14, 6))
    axes_flat = axes_p.ravel()

    for idx, (key, label, true_val) in enumerate(phys_params):
        ax = axes_flat[idx]
        for mname, (res, color) in sampling_results.items():
            if key in res.samples:
                vals = np.array(res.samples[key])
                ax.hist(vals, bins=min(20, len(vals)//2 + 1), color=color,
                        alpha=0.5, density=True, label=mname)
        ax.axvline(true_val, color="k", ls="--", lw=1.5, label="Truth")
        ax.set_xlabel(label, fontsize=10)
        ax.set_yticks([])
        if idx == 0:
            ax.legend(fontsize=7)

    # Hide extra subplot
    axes_flat[-1].set_visible(False)

    fig.suptitle("Posterior Distributions (Physical Parameters)", fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print("No sampling methods available — install blackjax or nifty8[re]")

### Corner Plot (geoVI)

The corner plot shows all pairwise correlations between physical parameters and derived quantities (stellar mass, SFR). Dashed lines mark the true values.

In [ ]:
if geovi_ok:
    # Add derived truths for corner plot
    d_true = model.predict_derived(true_params)
    corner_truths = dict(true_params)
    corner_truths["stellar_mass"] = float(d_true["stellar_mass"])
    corner_truths["sfr_100myr"] = float(d_true["sfr_100myr"])

    fig = result_geovi.plot_corner(truths=corner_truths)
    fig.suptitle("geoVI Posterior Corner Plot (Smooth Model)", fontsize=13, y=1.01)
    plt.show()
else:
    print("Skipping — geoVI not available")

---
## 6. Parameter Recovery Table

In [ ]:
print(f"{'Parameter':<22s} {'True':>8s}", end="")
for name in methods:
    print(f"  {name:>12s}", end="")
print()
print("-" * (22 + 10 + 14 * len(methods)))

for key in ["sfh_alpha", "sfh_beta", "sfh_tau_peak_gyr", "sfh_peak_sfr",
            "met_logzsol", "dust_tau_bc", "dust_tau_diff"]:
    true_v = true_params[key]
    print(f"{key:<22s} {true_v:>8.3g}", end="")
    for name, (res, _) in methods.items():
        s = res.summary()
        if key in s:
            if "median" in s[key]:
                print(f"  {s[key]['median']:>12.3g}", end="")
            else:
                print(f"  {s[key]['value']:>12.3g}", end="")
    print()

print()
print("Wall times:")
for name, (res, _) in methods.items():
    print(f"  {name}: {res.wall_time_s:.1f}s")

---
## 7. Derived Quantities

In [ ]:
d_true = model.predict_derived(true_params)

print(f"{'Quantity':<18s} {'True':>12s}", end="")
for name in methods:
    print(f"  {name:>12s}", end="")
print()
print("-" * (18 + 14 + 14 * len(methods)))

for key in ["stellar_mass", "sfr_100myr", "ssfr"]:
    print(f"{key:<18s} {float(d_true[key]):>12.3e}", end="")
    for name, (res, _) in methods.items():
        d = model.predict_derived(res.params)
        print(f"  {float(d[key]):>12.3e}", end="")
    print()

---
## 8. Stochastic Model: geoVI vs NUTS

The stochastic SFH model adds a GP with `n_grid=128` latent variables, making the total dimensionality 128 + 9 = 137. NUTS struggles in this regime (many divergences, slow mixing). geoVI handles it naturally because it approximates the posterior in a transformed Gaussian space.

This is the recommended workflow for the full IFT model: **MAP initialization → geoVI posterior**.

In [ ]:
# Stochastic model: GP with n_grid=128
spec_stoch = ParamSpec(
    sfh_alpha        = Uniform(0.5, 3.0),
    sfh_beta         = Uniform(0.3, 2.0),
    sfh_tau_peak_gyr = Uniform(0.5, 10.0),
    sfh_peak_sfr     = Uniform(0.1, 50.0),
    psd_sigma        = Uniform(0.1, 3.0),
    psd_tau_myr      = Uniform(1.0, 300.0),
    met_logzsol      = Uniform(-1.5, 0.2),
    dust_tau_bc      = Uniform(0.0, 3.0),
    dust_tau_diff    = Uniform(0.0, 2.0),
    dust_slope       = -0.7,
    redshift         = 0.1,
    stochastic       = True,
    n_grid           = 128,
)
model_stoch = Model(spec_stoch, ssp, filters=filters)

# True stochastic galaxy
true_stoch = {
    "sfh_alpha": 1.5, "sfh_beta": 1.0,
    "sfh_tau_peak_gyr": 4.0, "sfh_peak_sfr": 8.0,
    "psd_sigma": 1.2, "psd_tau_myr": 50.0,
    "met_logzsol": -0.3,
    "dust_tau_bc": 1.0, "dust_tau_diff": 0.3,
    "dust_slope": -0.7, "redshift": 0.1,
    "psd_xi": jax.random.normal(jax.random.PRNGKey(42), shape=(128,)),
}

mock_stoch = model_stoch.mock(true_stoch, snr=20.0, key=jax.random.PRNGKey(1))
sfh_true_stoch = model_stoch.predict_sfh(true_stoch)

print(f"Stochastic model: {spec_stoch.n_free} free + {spec_stoch.n_grid} psd_xi = "
      f"{spec_stoch.n_free + spec_stoch.n_grid} total dimensions")

In [ ]:
# MAP initialization
fitter_stoch = Fitter(model_stoch, mock_stoch.flux_obs, mock_stoch.noise)
result_map_s = fitter_stoch.run("map", n_steps=2000, learning_rate=0.03, verbose=True)
print(f"MAP: {result_map_s.wall_time_s:.1f}s")

In [ ]:
# geoVI (recommended for stochastic)
try:
    old_stdout = sys.stdout
    sys.stdout = io.StringIO()
    result_geovi_s = fitter_stoch.run(
        "geovi", init_from=result_map_s,
        n_iterations=15, n_samples=6, n_posterior_samples=40, verbose=False,
    )
    sys.stdout = old_stdout
    geovi_stoch_ok = True
    print(f"geoVI: {result_geovi_s.diagnostics['n_samples']} samples "
          f"in {result_geovi_s.wall_time_s:.1f}s")
except Exception as e:
    sys.stdout = old_stdout
    geovi_stoch_ok = False
    print(f"geoVI failed: {e}")

In [ ]:
# NUTS (for comparison — expect divergences)
try:
    result_nuts_s = fitter_stoch.run(
        "nuts", init_from=result_map_s,
        n_warmup=100, n_samples=50, verbose=True,
    )
    nuts_stoch_ok = True
    print(f"\nNUTS: {result_nuts_s.diagnostics['n_divergent']}/{result_nuts_s.diagnostics['n_samples']} divergences, "
          f"{result_nuts_s.wall_time_s:.1f}s")
except Exception as e:
    nuts_stoch_ok = False
    print(f"NUTS failed: {type(e).__name__}: {str(e)[:200]}")

### Stochastic SFH Recovery (Linear Scale)

With 137 dimensions, geoVI finds the posterior efficiently while NUTS struggles. The SFH is plotted in linear scale.

In [ ]:
methods_s = {"MAP": (result_map_s, "C0")}
if geovi_stoch_ok:
    methods_s["geoVI"] = (result_geovi_s, "C2")
if nuts_stoch_ok:
    methods_s["NUTS"] = (result_nuts_s, "C1")

n_m = len(methods_s)
fig, axes = plt.subplots(1, n_m, figsize=(5 * n_m, 5), sharey=True)
if n_m == 1: axes = [axes]

for ax, (name, (res, color)) in zip(axes, methods_s.items()):
    model_stoch.plot_sfh_posterior(res, true_params=true_stoch, ax=ax,
                                  color=color, label=name)
    ax.set_title(f"{name} — Stochastic ({res.wall_time_s:.1f}s)")

plt.suptitle(f"Stochastic SFH Recovery (n_grid={spec_stoch.n_grid}, 16-84% fill)",
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print("\nTiming comparison (stochastic model):")
for name, (res, _) in methods_s.items():
    extra = ""
    if 'n_divergent' in res.diagnostics:
        extra = f", {res.diagnostics['n_divergent']} divergences"
    print(f"  {name:6s}: {res.wall_time_s:6.1f}s{extra}")

### Key Observation

For the stochastic model (128 GP dimensions + 9 physical parameters = 137D):
- **geoVI** handles it naturally — the Gaussian approximation in the transformed coordinate system is efficient
- **NUTS** struggles with many divergences and slow convergence — the high-dimensional GP space makes HMC trajectories unstable

This is why **geoVI is the recommended inference method** for the full IFT model with stochastic SFHs.

---
## Key Takeaways

1. **MAP** is fastest (seconds) — good for catalogs, no uncertainties
2. **NUTS** gives exact posteriors (minutes) — 10-100x faster than dynesty/MultiNest
3. **geoVI** gives approximate posteriors (seconds) — best speed/accuracy trade-off
4. All three use the **same loss function** and benefit from end-to-end JAX gradients
5. **Photometry alone** constrains the broad SED shape but not individual SFH parameters — spectroscopy helps break degeneracies